# D2: champion squads against the season's top 15 scorers

> Compare the distribution of experience of active players on the champion team, and
> their height, over the last two seasons, with the experience and height distribution
> of the top 15 players of that season.

Two seasons, two groups in each, two measurements per player. Every word in that
sentence has to be pinned down before any number means anything, so that is where
this starts.

## Reading the question

"The last two seasons" are 2024-25 and 2025-26, held in the database as `2025` and `2026`.
Oklahoma City won the first, New York the second. The most recent scrape brought 2025-26 in
as a finished season, so this window sits a year later than the one the original bootcamp
analysis used.

"Active players on the champion team" means on the champion's roster *and* on court for at
least one game. A roster spot and an appearance are not the same claim. The second condition
turns out to exclude almost nobody, 18 players in 2024-25 and 17 in 2025-26, but the two are
worth separating anyway.

"The top 15 players of that season" means the 15 highest point scorers. This database
carries no wins, no minutes-weighted rating, no all-round score. Scoring volume is the order
Basketball-Reference's own season pages use, so it is the order used here. The group is "the
season's 15 biggest scorers", not "the season's 15 best players", and every sentence below
keeps that distinction.

"Experience" is seasons played before the one in question. A rookie is `0`.

Two further decisions. The groups overlap by one player in each season, because the
champion's leading scorer also lands in the league's top 15: Shai Gilgeous-Alexander in
2024-25, Jalen Brunson in 2025-26. The question names two populations rather than cutting
the league in half, so he belongs in both and stays in both. It does mean the two samples in
a season are not quite independent.

The seasons are then compared separately rather than pooled. Pooling puts 35 players from
two different clubs into one box. Oklahoma City fielded a notably young squad and New York
did not, so a pooled average would sit between two teams and describe neither. Each season
gets its own comparison and the pair is read side by side.

One structural point to carry through to the end. A champion roster is an entire squad: the
leading scorer, the rotation behind him, and the rookies at the end of the bench. The top 15
scorers are 15 leading men drawn from 15 different clubs. The groups are built by different
rules, so a gap between them describes squads against stars at least as much as it describes
champions.

## What the data has to supply

For each of the two seasons we need two lists of people.

The first list is everyone who was signed to the championship-winning club that season and
got onto the court at least once. The second is the fifteen players who scored the most
points in the whole league that season, whichever club they played for.

For every player on either list we need two facts: how tall he is, and how many seasons he
had already played before that one started.

Height comes off the player's own profile page, it is recorded the same way for everybody,
and nobody in either group is missing it. It is coarser than it looks, though. The source
publishes height in whole inches, so what arrives is a ladder of values about 2.5 cm apart
rather than a smooth measurement, and a difference smaller than one rung is smaller than the
measurement itself.

Experience is the awkward one. It sits in two places in this database, collected from two
different pages, and only one of them states it outright. The other works it out backwards
from a career total. So before comparing anything, the first job is to check whether the two
places agree.

In [1]:
import _setup  # noqa: F401

import pandas as pd

import utils.custom_plots as cp
import utils.custom_stats as cs
from utils.db_utils import run_query

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

# The window this question is asked over: 2024-25 and 2025-26.
FIRST_SEASON = 2025
LAST_SEASON = 2026
METRICS = ["experience_seasons", "height_cm"]

In [2]:
SQL_GROUPS = """
with roster_experience as (
    -- The experience figure the roster page states outright, one row per
    -- player-season. A player traded mid-season has a roster row at each club
    -- and both carry the same figure, so max() only collapses the duplicate.
    select season,
           player_id,
           max(experience_seasons) as experience_roster
    from processed.rosters
    where season between :first_season and :last_season
    group by season, player_id
),
group_members as (
    select season, player_id, 'champion' as group_kind
    from analyst_ready.player_season
    where season between :first_season and :last_season
      and is_on_champion_team
      and games_played > 0
    union all
    select season, player_id, 'top15' as group_kind
    from analyst_ready.player_season
    where season between :first_season and :last_season
      and points_rank <= 15
)
select ps.season,
       ps.season_label,
       ds.champion_team_name,
       gm.group_kind,
       ps.player_id,
       ps.player_name,
       ps.team_name,
       ps.position,
       ps.points_rank,
       ps.games_played,
       ps.height_cm,
       re.experience_roster,
       ps.experience_seasons as experience_rolled_back
from group_members gm
join analyst_ready.player_season ps
  on ps.season = gm.season
 and ps.player_id = gm.player_id
join analyst_ready.dim_season ds
  on ds.season = ps.season
left join roster_experience re
  on re.season = gm.season
 and re.player_id = gm.player_id
order by ps.season, gm.group_kind, ps.player_name
"""

raw = run_query(
    SQL_GROUPS,
    {"first_season": FIRST_SEASON, "last_season": LAST_SEASON},
)

# SQL numeric arrives as Decimal typed object; custom_plots skips those silently.
for col in ("height_cm", "experience_roster", "experience_rolled_back"):
    raw[col] = raw[col].astype(float)

print(raw.shape)
print(raw.groupby(["season_label", "group_kind"]).size())
raw.head()

(65, 13)
season_label  group_kind
2024-25       champion      18
              top15         15
2025-26       champion      17
              top15         15
dtype: int64


,season,season_label,champion_team_name,group_kind,player_id,player_name,team_name,position,points_rank,games_played,height_cm,experience_roster,experience_rolled_back
0,2025,2024-25,Oklahoma City Thunder,champion,wiggiaa01,Aaron Wiggins,Oklahoma City Thunder,SG,98,76,195.6,3.0,3.0
1,2025,2024-25,Oklahoma City Thunder,champion,flaglad01,Adam Flagler,Oklahoma City Thunder,SG,460,37,185.4,1.0,0.0
2,2025,2024-25,Oklahoma City Thunder,champion,mitchaj01,Ajay Mitchell,Oklahoma City Thunder,SG,339,36,193.0,0.0,0.0
3,2025,2024-25,Oklahoma City Thunder,champion,carusal01,Alex Caruso,Oklahoma City Thunder,SG,276,54,195.6,7.0,7.0
4,2025,2024-25,Oklahoma City Thunder,champion,ducasal01,Alex Ducas,Oklahoma City Thunder,SG,487,21,200.7,0.0,NaN


## Two sources for experience, and whether they agree

The roster page prints a player's experience next to his name for that season. That figure
is stated, not computed, and it lives in `processed.rosters.experience_seasons`.

The player-season table has its own `experience_seasons`, and that one is worked out
backwards: take the career total from the profile page and subtract the number of seasons
since. The arithmetic is exact only for a player who appeared in every season in between.
Anyone who missed a full year comes out too low, and the error grows the further back you
look.

The original framing of this question assumed the two groups would have to use different
sources, because the roster pages were scraped for champions only. That is no longer true.
The current scrape covers all 30 clubs from 2018-19 onward, so every player in both groups
has a stated figure available, and the derived one can be treated as something to check
against rather than something to rely on.

The check below covers three things: whether every player in either group has a stated
figure, whether the two match where both exist, and which one is wrong where they do not.

In [3]:
# One row per player-season: the overlap player would otherwise be counted twice.
players = raw.drop_duplicates(["season", "player_id"]).copy()
players["comparable"] = (
    players["experience_roster"].notna() & players["experience_rolled_back"].notna()
)
players["sources_agree"] = players["comparable"] & players["experience_roster"].eq(
    players["experience_rolled_back"]
)

coverage = players.groupby(["season_label", "group_kind"]).agg(
    players=("player_id", "size"),
    stated=("experience_roster", "count"),
    derived=("experience_rolled_back", "count"),
    comparable=("comparable", "sum"),
    agree=("sources_agree", "sum"),
)
print(coverage)

mismatch = players.loc[
    players["comparable"] & ~players["sources_agree"],
    ["season_label", "group_kind", "player_name", "experience_roster",
     "experience_rolled_back"],
]
missing_derived = players.loc[
    players["experience_rolled_back"].isna(),
    ["season_label", "group_kind", "player_name", "experience_roster",
     "experience_rolled_back"],
]
pd.concat([mismatch, missing_derived]).sort_values(["season_label", "player_name"])

                         players  stated  derived  comparable  agree
season_label group_kind                                             
2024-25      champion         18      18       17          17     16
             top15            14      14       14          14     14
2025-26      champion         17      17       17          17     17
             top15            14      14       14          14     14


,season_label,group_kind,player_name,experience_roster,experience_rolled_back
1,2024-25,champion,Adam Flagler,1.0,0.0
4,2024-25,champion,Alex Ducas,0.0,NaN


### What the check found

All 63 player-seasons across the four groups have a stated figure from the roster page.
(63 rather than 65 because the player who appears in both groups is counted once here,
under `champion`.) So both groups can use the same source, and the difference in sourcing
that this question was expected to turn on does not arise.

The derived figure exists for 62 of them and matches the stated one on 61. The two
exceptions are both Oklahoma City players in 2024-25:

- **Adam Flagler**: stated 1 season of experience, derived 0. He is exactly the case the
  roll-back gets wrong, since a career total minus elapsed seasons only works if the player
  appeared in every season in between, and he did not.
- **Alex Ducas**: stated 0, derived missing entirely. His profile page carries no career
  experience figure, so there was nothing to roll back from.

Everyone in the top 15 agrees on both counts, in both seasons, which fits what the source
does. The newest seasons need little or no roll-back, and established scorers rarely miss a
whole year.

**Decision:** experience comes from the roster page for every player in both groups. It is
the stated figure rather than a reconstruction, it corrects one wrong value, and it fills
the one gap. The derived column stays in the data only as the thing that was checked
against.

In [4]:
groups = raw.copy()
groups["experience_seasons"] = groups["experience_roster"]

# One label per season-and-group, so both seasons sit in one figure.
groups["player_group"] = groups["season_label"] + " · " + groups["group_kind"].map(
    {"champion": "champion", "top15": "top 15 scorers"}
)
is_champion = groups["group_kind"].eq("champion")
groups.loc[is_champion, "player_group"] = (
    groups.loc[is_champion, "season_label"]
    + " · "
    + groups.loc[is_champion, "champion_team_name"]
)

analysis = groups[
    ["season", "season_label", "player_group", "group_kind", "player_name",
     "position", "points_rank", "games_played", *METRICS]
].sort_values(["season", "group_kind", "player_name"])

print(analysis["player_group"].value_counts().sort_index())
print(f"\nmissing values in the two metrics: {analysis[METRICS].isna().sum().sum()}")

# Both metrics land on a coarse grid, which matters at these group sizes.
for metric in METRICS:
    values = pd.Series(sorted(analysis[metric].unique()))
    print(f"{metric}: {len(values)} distinct values, "
          f"smallest step {values.diff().min():.2f}")

analysis.head()

player_group
2024-25 · Oklahoma City Thunder    18
2024-25 · top 15 scorers           15
2025-26 · New York Knicks          17
2025-26 · top 15 scorers           15
Name: count, dtype: int64

missing values in the two metrics: 0
experience_seasons: 16 distinct values, smallest step 1.00
height_cm: 13 distinct values, smallest step 2.50


,season,season_label,player_group,group_kind,player_name,position,points_rank,games_played,experience_seasons,height_cm
0,2025,2024-25,2024-25 · Oklahoma City Thunder,champion,Aaron Wiggins,SG,98,76,3.0,195.6
1,2025,2024-25,2024-25 · Oklahoma City Thunder,champion,Adam Flagler,SG,460,37,1.0,185.4
2,2025,2024-25,2024-25 · Oklahoma City Thunder,champion,Ajay Mitchell,SG,339,36,0.0,193.0
3,2025,2024-25,2024-25 · Oklahoma City Thunder,champion,Alex Caruso,SG,276,54,7.0,195.6
4,2025,2024-25,2024-25 · Oklahoma City Thunder,champion,Alex Ducas,SG,487,21,0.0,200.7


## The four groups, described

Four groups, 15 to 18 players each. At that size a summary table is close to the raw data
anyway, so the spread matters as much as the average: the minimum, the maximum and the
median say more about a 17-player squad than its mean does.

In [5]:
descriptives = pd.concat(
    [
        cs.summary_stats(sub[METRICS]).assign(group=name)
        for name, sub in analysis.groupby("player_group", sort=True)
    ],
    ignore_index=True,
)
descriptives[
    ["group", "column", "n", "mean", "median", "std", "min", "q1", "q3", "max", "skew"]
].round(2)

,group,column,n,mean,median,std,min,q1,q3,max,skew
0,2024-25 · Oklahoma City Thunder,experience_seasons,18,2.56,2.00,2.48,0.0,0.25,4.75,7.0,0.58
1,2024-25 · Oklahoma City Thunder,height_cm,18,199.81,196.85,8.54,185.4,193.65,205.70,215.9,0.50
2,2024-25 · top 15 scorers,experience_seasons,15,8.47,9.00,4.17,3.0,5.50,10.50,15.0,0.42
3,2024-25 · top 15 scorers,height_cm,15,198.46,195.60,7.85,188.0,194.30,200.65,213.4,0.77
4,2025-26 · New York Knicks,experience_seasons,17,4.59,4.00,3.68,0.0,1.00,7.00,11.0,0.28
5,2025-26 · New York Knicks,height_cm,17,200.38,198.10,8.65,188.0,195.60,205.70,213.4,0.14
6,2025-26 · top 15 scorers,experience_seasons,15,9.47,9.00,3.56,5.0,7.00,10.50,17.0,0.96
7,2025-26 · top 15 scorers,height_cm,15,197.95,198.10,7.57,188.0,193.00,203.20,210.8,0.34


Experience separates the groups immediately. Oklahoma City's title squad averaged 2.6
seasons of experience, against 8.5 for that season's top 15 scorers, a gap of almost six
seasons. New York's squad averaged 4.6 against 9.5, a gap of just under five. The ranges
barely meet: both champion squads carried rookies at 0, while the least experienced top-15
scorer had 3 seasons behind him in 2024-25 and 5 in 2025-26.

Height does not separate them. The four means sit between 198.0 cm and 200.4 cm, and the
champion squad is the taller of the pair in both seasons, by 1.4 cm in 2024-25 and 2.4 cm in
2025-26. Standard deviations run near 8 cm in every group, so those gaps are a quarter of a
standard deviation and less. Every group spans roughly 185 cm to 215 cm, which is simply the
range of an NBA squad.

Both gaps also run into the resolution of the measurement. All 63 players in this study take
one of 13 heights, each about 2.5 cm from the next, because the source records height in
whole inches. A mean difference of 1.4 cm is half a rung of that ladder. The 2.4 cm gap is
just under one rung. Neither is a difference the data can really hold.

In [6]:
fig_experience = cp.grouped_box_plot(
    analysis,
    group_col="player_group",
    value_col="experience_seasons",
    show_points="all",
    sort_by="n",
    orientation="horizontal",
    title="Experience: champion squad vs the season's top 15 scorers",
)
fig_experience

Every group carries the gold low-n outline, because every group is under 30 players. That is
not a fault in the data. A champion roster is about 17 people and the top 15 is 15 by
definition. It is the reason the rest of this notebook leans on intervals rather than
p-values.

The two champion boxes sit almost entirely to the left of the two top-15 boxes. Oklahoma
City's whole squad fits inside 0 to 7 seasons, which is roughly where the top 15 *starts*.
New York's squad stretches further right but still has its median at 4 against 9. The
overlap between a champion squad and the league's top scorers is the handful of veterans
each squad carries, and in Oklahoma City's case that meant Alex Caruso, at 7 seasons, being
the most experienced player on a title-winning team.

In [7]:
fig_height = cp.grouped_box_plot(
    analysis,
    group_col="player_group",
    value_col="height_cm",
    show_points="all",
    sort_by="n",
    orientation="horizontal",
    title="Height: champion squad vs the season's top 15 scorers",
)
fig_height

Height tells the opposite story: four boxes covering much the same ground. Each group runs
from around 188 cm to around 213 cm, the medians sit within 2.5 cm of each other, and the
interquartile ranges overlap almost completely. Oklahoma City's squad is the widest of the
four, holding both the shortest player in the study at 185.4 cm and the tallest at 215.9 cm.

The strip of points under each box shows how few distinct heights there are. Players stack
on top of each other at each rung, and the box hinges land on rungs too. That is worth
knowing before reading the whiskers: the 2024-25 top 15 has its interquartile range squeezed
into about 6 cm, which pushes the outlier fence down to 210.2 cm and puts Giannis
Antetokounmpo, Nikola Jokić and Karl-Anthony Towns outside it. Three of the league's best
centres are not statistical anomalies. They are one or two rungs above a group whose middle
happens to be tightly packed.

The small edge the champions hold has an ordinary explanation. A full roster carries four
centres, while a list of the league's 15 biggest scorers carried two in 2024-25 and one in
2025-26 and leaned toward guards instead. That is squad composition rather than a height
advantage that wins titles.

In [8]:
fig_ecdf = cp.ecdf_plot(
    analysis,
    cols="experience_seasons",
    group_col="player_group",
    mark_percentiles=[0.25, 0.5, 0.75],
    title="Experience, cumulative: share of each group at or below x seasons",
)
fig_ecdf

The cumulative view says the same thing without depending on a mean. Each curve reads as
"what share of this group had at most this many seasons behind them". Both champion curves
sit above and to the left of both top-15 curves at every point on the axis, which is as
clean a separation as this kind of comparison produces: pick any experience threshold and a
larger share of the champion squad falls below it than of the top 15.

The left edges are the sharpest detail. A quarter of Oklahoma City's squad had never played
an NBA season before that year, and a third of New York's had at most one. The top-15 curves
do not start until 3 seasons in 2024-25 and 5 in 2025-26.

## How much of this survives the sample size

Fifteen to eighteen players is a small group. Two things follow. A group mean is not pinned
down tightly, so it needs an interval around it rather than a decimal place. And a p-value
computed on 33 people is a blunt instrument, so the interval around the *difference*
carries more information than the verdict attached to it.

Both are below: first a bootstrap interval on each group's mean, then a test of each
season's gap with a bootstrap interval on the difference.

In [9]:
group_means = pd.concat(
    [
        cs.bootstrap_ci(analysis, value_col=metric, group_col="player_group")
        .assign(metric=metric)
        for metric in METRICS
    ],
    ignore_index=True,
)
group_means[
    ["metric", "group", "n", "estimate", "ci_low", "ci_high", "se", "flags"]
].round(2)

,metric,group,n,estimate,ci_low,ci_high,se,flags
0,experience_seasons,2024-25 · Oklahoma City Thunder,18,2.56,1.56,3.78,0.57,small n (18): interval is optimistic
1,experience_seasons,2024-25 · top 15 scorers,15,8.47,6.60,10.67,1.05,small n (15): interval is optimistic
2,experience_seasons,2025-26 · New York Knicks,17,4.59,2.94,6.29,0.87,small n (17): interval is optimistic
3,experience_seasons,2025-26 · top 15 scorers,15,9.47,7.93,11.53,0.89,small n (15): interval is optimistic
4,height_cm,2024-25 · Oklahoma City Thunder,18,199.81,196.15,203.90,1.95,small n (18): interval is optimistic
5,height_cm,2024-25 · top 15 scorers,15,198.46,195.23,202.86,1.95,small n (15): interval is optimistic
6,height_cm,2025-26 · New York Knicks,17,200.38,196.35,204.41,2.04,small n (17): interval is optimistic
7,height_cm,2025-26 · top 15 scorers,15,197.95,194.40,201.83,1.89,small n (15): interval is optimistic


Every row carries the same warning, and it is the right one: at n between 15 and 18 a
bootstrap interval is optimistic, because resampling can only ever draw values that were
already observed. Nothing worse than that showed up. Both metrics are heavily tied, which
can leave a BCa interval undefined, and the toolkit would have flagged it here. All eight
intervals came back finite.

Taken at face value the experience intervals do not come close to touching. Oklahoma City's
squad mean lands between 1.6 and 3.8 seasons against 6.6 to 10.7 for the 2024-25 top 15;
New York's between 2.9 and 6.3 against 7.9 to 11.5. The height intervals do the reverse and
overlap across nearly their whole width in both seasons.

In [10]:
# The champion label sorts first within each season, so it is group A throughout:
# a positive estimate means the champion squad sits above the top 15.
comparisons = pd.concat(
    [
        cs.compare_groups(
            analysis.loc[analysis["season"] == season],
            group_col="player_group",
            value_col=metric,
            test="auto",
            ci="bootstrap",
        ).assign(metric=metric)
        for season in (FIRST_SEASON, LAST_SEASON)
        for metric in METRICS
    ],
    ignore_index=True,
)
comparisons[
    ["metric", "comparison", "group_sizes", "test", "chosen_because", "estimate_type",
     "estimate", "ci_low", "ci_high", "p_value", "decision", "effect_size",
     "effect_type", "magnitude", "flags"]
].round(3)

,metric,comparison,group_sizes,test,chosen_because,estimate_type,estimate,ci_low,ci_high,p_value,decision,effect_size,effect_type,magnitude,flags
0,experience_seasons,2024-25 · Oklahoma City Thunder vs 2024-25 · t...,2024-25 · Oklahoma City Thunder=18; 2024-25 · ...,mannwhitney,"auto: normal 1/2, variances equal",median difference,-7.000,-10.000,-4.000,0.000,reject H₀,-0.789,cliffs_delta,large,small group (n=15)
1,height_cm,2024-25 · Oklahoma City Thunder vs 2024-25 · t...,2024-25 · Oklahoma City Thunder=18; 2024-25 · ...,welch,"auto: normal 2/2, variances equal",mean difference,1.351,-4.071,6.623,0.640,fail to reject H₀,0.160,hedges_g,negligible,small group (n=15)
2,experience_seasons,2025-26 · New York Knicks vs 2025-26 · top 15 ...,2025-26 · New York Knicks=17; 2025-26 · top 15...,mannwhitney,"auto: normal 1/2, variances equal",median difference,-5.000,-9.000,0.000,0.002,reject H₀,-0.627,cliffs_delta,large,small group (n=15)
3,height_cm,2025-26 · New York Knicks vs 2025-26 · top 15 ...,2025-26 · New York Knicks=17; 2025-26 · top 15...,welch,"auto: normal 2/2, variances equal",mean difference,2.430,-2.952,7.905,0.404,fail to reject H₀,0.290,hedges_g,small,small group (n=15)


The champion squad is group A in both seasons, so a negative estimate means the champions
sit below the top 15 and a positive one means above.

`test='auto'` screened each pair and split the two metrics. Experience is not normal in the
champion squads, which hold a cluster of rookies at zero and a thin tail of veterans, so it
went to Mann-Whitney and reports a median difference. Height passed the normality screen in
all four groups and went to Welch, reporting a mean difference.

**Experience.** The champion squad's median sits 7 seasons below the top 15 in 2024-25
(p = 0.0001) and 5 seasons below in 2025-26 (p = 0.002). Cliff's delta is −0.79 and −0.63,
large in both seasons. Delta is a statement about pairs: draw one player from each group and
the top-15 scorer is the more experienced in the large majority of those draws.

One honest wrinkle. The bootstrap interval on the 2025-26 median difference runs from −9 to
0, so its upper edge touches no difference at all even though the test rejects comfortably.
Experience is a small integer with heavy ties, so a bootstrapped median lands on one of a
handful of values rather than moving smoothly, which makes that interval coarse. The 2024-25
interval, −10 to −4, has no such problem.

**Height.** The champions are 1.4 cm taller in 2024-25 (p = 0.64) and 2.4 cm taller in
2025-26 (p = 0.40). Both intervals span zero by a wide margin, roughly −4 to +7 cm and −3 to
+8 cm, and the effect sizes are 0.16 and 0.29. Two things are working against these tests at
once: 15 to 18 players is not enough to detect a gap this small, and a gap this small is at
or below the 2.5 cm step the heights are recorded in. The conclusion is that this data
cannot separate the two groups on height, not that they are identical.

## Conclusion

**Experience separates the two groups. Height does not.**

Across both seasons the champion squad was the less experienced group by a wide margin.
Oklahoma City's 2024-25 title squad averaged 2.6 seasons of prior NBA experience against 8.5
for that season's 15 leading scorers. New York's 2025-26 squad averaged 4.6 against 9.5.
Median gaps of 7 and 5 seasons, large effect sizes in both, and group means whose intervals
do not overlap. The cumulative curves make the same point without relying on an average: at
every experience threshold, more of the champion squad sits below it.

On height the two groups are not distinguishable in this data. The champions were 1.4 cm and
2.4 cm taller on average, with intervals running roughly ±5 cm around the difference. Both
gaps are also at or below the 2.5 cm step the source records height in, so there is no
finding here to defend either way. What difference there is tracks squad composition, since
a title roster carries four centres and a top-15 scoring list carried one or two.

**What is actually being compared.** The two groups are built by different rules. A champion
roster is an entire squad, from the leading scorer down to a rookie who played 21 games. The
top 15 is 15 established scorers, one or two per club, selected for the thing that takes
years to become good at. Most of the experience gap is that structural difference rather
than evidence that inexperienced teams win titles. The honest reading is that a title-winning
squad is mostly *not* made of the league's top scorers, and the players filling out those
rosters are young.

**Limits worth stating.** Two seasons, two clubs, 15 to 18 players a side. Each squad is a
single draw from the way one front office chose to build one team, and the two differ from
each other: Oklahoma City's was the younger by two seasons on average, which is most of why
the 2024-25 gap is the larger of the two. Widening the window to more champions would settle
whether this is a pattern or two data points that happen to agree. And "top 15" here is
scoring volume, so all of this describes the league's biggest scorers rather than its best
players.